# PuLID-FLUX modular — E2E validation (for #10141)

Loads `remyxai/pulid-flux-modular` via `trust_remote_code` (block.py fetches the vendored `eva_clip` at
runtime), then validates **training-free identity personalization**:
1. same prompt/seed with `id_weight=0` (no-op = stock FLUX) vs `id_weight=1` (personalized),
2. an **ArcFace cosine** metric (reference face vs generated face) across `id_weight ∈ {0, 0.5, 1.0}` —
   identity similarity should rise with the weight.

Runtime: A100/L4 · `HUGGINGFACE_TOKEN` · accept **FLUX.1-dev** license.

## 1 · Install

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf
!pip install -q insightface onnxruntime-gpu facexlib timm einops ftfy opencv-python

## 2 · GPU + auth

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16

## 3 · A reference face
An AI-generated face (no real identity) — swap in your own `id.png` to personalize on it.

In [ ]:
import numpy as np, requests
from PIL import Image
from io import BytesIO
try:
    r = requests.get("https://thispersondoesnotexist.com", timeout=30,
                     headers={"User-Agent": "Mozilla/5.0"})
    ref = Image.open(BytesIO(r.content)).convert("RGB")
except Exception as e:
    print("fallback:", e)
    r = requests.get("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/input_image_vermeer.png", timeout=30)
    ref = Image.open(BytesIO(r.content)).convert("RGB")
ref.save("ref.png"); print("reference:", ref.size)
from IPython.display import display; display(ref.resize((256,256)))

## 4 · Load the modular pipeline (trust_remote_code)

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained("remyxai/pulid-flux-modular", trust_remote_code=True)
print("loaded block:", type(pipe.blocks).__name__)   # expect PuLIDFluxBlock
pipe.load_components(dtype=DT); pipe.to(DEV)
print("components loaded")

## 5 · Personalized vs stock (same prompt + seed)

In [ ]:
from IPython.display import display
PROMPT = "portrait of a person as an astronaut, cinematic lighting, detailed space suit"
def gen(id_weight, seed=0):
    g = torch.Generator(DEV).manual_seed(seed)
    return pipe(prompt=PROMPT, id_image=ref, id_weight=id_weight, height=1024, width=1024,
                num_inference_steps=20, guidance_scale=4.0, generator=g).images[0]
img0 = gen(0.0); img0.save("pulid_w0.png")     # id_weight=0 -> stock FLUX (no-op)
img1 = gen(1.0); img1.save("pulid_w1.png")     # id_weight=1 -> personalized
from PIL import Image
comp = Image.new("RGB", (1024*3+40, 1024), "white")
comp.paste(ref.resize((1024,1024)), (0,0)); comp.paste(img0,(1024+20,0)); comp.paste(img1,(2048+40,0))
comp.save("pulid_ref_vs_stock_vs_id.png")
print("left=reference · middle=id_weight0 (stock) · right=id_weight1 (personalized)")
display(comp.resize((1050, 350)))

## 6 · Quantitative: ArcFace identity similarity vs id_weight

In [ ]:
import numpy as np, cv2
from insightface.app import FaceAnalysis
app = FaceAnalysis(name="antelopev2", root=".", providers=["CUDAExecutionProvider","CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640,640))
def embed(pil):
    faces = app.get(cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR))
    if not faces: return None
    f = sorted(faces, key=lambda x:(x["bbox"][2]-x["bbox"][0])*(x["bbox"][3]-x["bbox"][1]))[-1]
    e = f["embedding"]; return e/np.linalg.norm(e)
ref_e = embed(ref)
print("id_weight | ArcFace cosine(reference, generated)")
rows=[]
for w in [0.0, 0.5, 1.0]:
    im = gen(w, seed=1); im.save(f"pulid_sim_w{w}.png")
    ge = embed(im)
    cos = float(np.dot(ref_e, ge)) if (ref_e is not None and ge is not None) else float("nan")
    rows.append((w,cos)); print(f"   {w:>3}    |  {cos:.3f}")
ok = rows[-1][1] > rows[0][1] + 0.1
print("\nidentity increases with id_weight:", "PASS" if ok else "REVIEW", rows)

## Verdict
- `loaded block: PuLIDFluxBlock` + a coherent personalized portrait (right) that clearly differs from the
  stock middle image = the injection + eva_clip runtime-load path work end-to-end.
- ArcFace cosine rising with `id_weight` = identity is actually transferred (not just a random face).
Then: publish public + draft the #10141 reply.